In [1]:
"""
So Sánh Các Chiến Lược Điền Khuyết Dữ Liệu
"""

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
# IterativeImputer hiện vẫn nằm trong module experimental của sklearn
from sklearn.experimental import enable_iterative_imputer  # Bắt buộc phải import dòng này trước
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

# 1. Tạo tập dữ liệu mô phỏng có tương quan thực tế
# Logic: Lương (Salary) phụ thuộc tuyến tính vào Số năm kinh nghiệm (Experience)
np.random.seed(42)
n = 10

experience = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0])
# Công thức gốc: Salary = Experience * 1500 + 5000 + Noise
salary = experience * 1500 + 5000 + np.random.normal(0, 500, size=n)

# Cố tình tạo khuyết thiếu theo cơ chế MAR
experience[3] = np.nan  # Khuyết ở dòng index 3
salary[3] = 11000.0     # Dòng này có salary nhưng thiếu exp
salary[7] = np.nan      # Dòng này có exp=8 nhưng thiếu salary

df_sample = pd.DataFrame({
    "Experience": experience,
    "Salary": salary
})

print("=== DỮ LIỆU GỐC CHỨA KHUYẾT THIẾU ===")
print(df_sample)

# 2. Điền khuyết cơ bản: Mean vs Median
mean_imputer = SimpleImputer(strategy="mean")
df_mean = pd.DataFrame(mean_imputer.fit_transform(df_sample), columns=df_sample.columns)

median_imputer = SimpleImputer(strategy="median")
df_median = pd.DataFrame(median_imputer.fit_transform(df_sample), columns=df_sample.columns)

# 3. Điền khuyết nâng cao: KNNImputer (K=2)
knn_imputer = KNNImputer(n_neighbors=2, weights="distance")
df_knn = pd.DataFrame(knn_imputer.fit_transform(df_sample), columns=df_sample.columns)

# 4. Điền khuyết nâng cao: IterativeImputer (MICE với Bayesian Ridge)
mice_imputer = IterativeImputer(estimator=BayesianRidge(), max_iter=20, random_state=42)
df_mice = pd.DataFrame(mice_imputer.fit_transform(df_sample), columns=df_sample.columns)

# 5. Tạo cờ báo khuyết (Missing Indicator)
indicator = MissingIndicator(features="missing-only")
missing_flags = indicator.fit_transform(df_sample)
flag_cols = [f"{col}_is_missing" for col in df_sample.columns[indicator.features_]]
df_flags = pd.DataFrame(missing_flags, columns=flag_cols)

# 6. Tổng hợp so sánh kết quả tại các dòng bị khuyết (Dòng 3 và Dòng 7)
comparison = pd.DataFrame({
    "Original_Exp": df_sample["Experience"],
    "Mean_Exp": df_mean["Experience"].round(1),
    "KNN_Exp": df_knn["Experience"].round(1),
    "MICE_Exp": df_mice["Experience"].round(1),
    "Original_Sal": df_sample["Salary"],
    "Mean_Sal": df_mean["Salary"].round(1),
    "KNN_Sal": df_knn["Salary"].round(1),
    "MICE_Sal": df_mice["Salary"].round(1),
})

print("\n=== ĐỐI CHIẾU KẾT QUẢ ĐIỀN KHUYẾT TẠI CÁC DÒNG 3 VÀ 7 ===")
print(comparison.iloc[[3, 7]])

print("\n=== MA TRẬN CỜ BÁO KHUYẾT (MISSING INDICATORS) ===")
print(df_flags.iloc[[3, 7]])

=== DỮ LIỆU GỐC CHỨA KHUYẾT THIẾU ===
   Experience        Salary
0         1.0   6748.357077
1         2.0   7930.867849
2         3.0   9823.844269
3         NaN  11000.000000
4         5.0  12382.923313
5         6.0  13882.931522
6         7.0  16289.606408
7         8.0           NaN
8         9.0  18265.262807
9        10.0  20271.280022

=== ĐỐI CHIẾU KẾT QUẢ ĐIỀN KHUYẾT TẠI CÁC DÒNG 3 VÀ 7 ===
   Original_Exp  Mean_Exp  KNN_Exp  MICE_Exp  Original_Sal  Mean_Sal  KNN_Sal  \
3           NaN       5.7      3.9       3.9       11000.0   11000.0  11000.0   
7           8.0       8.0      8.0       8.0           NaN   12955.0  17277.4   

   MICE_Sal  
3   11000.0  
7   17134.9  

=== MA TRẬN CỜ BÁO KHUYẾT (MISSING INDICATORS) ===
   Experience_is_missing  Salary_is_missing
3                   True              False
7                  False               True


In [2]:
"""
Trắc Lượng & Xử Lý Ngoại Lai: IQR vs Z-Score vs Modified Z-Score
 DUT AI Board - Data Science & Machine Learning Course
"""

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

# 1. Khởi tạo dữ liệu thu nhập mô phỏng (gồm phân phối chuẩn + ngoại lai cực đoan)
np.random.seed(42)
normal_income = np.random.normal(loc=1500, scale=300, size=100) # Lương thông thường ($1500 +- 300)
# Cố tình đưa vào các ngoại lai lỗi và ngoại lai tự nhiên
outliers_error = np.array([-500, 99999])  # Lỗi số âm và gõ thừa số 9
outliers_legit = np.array([5500, 6200])   # Lương giám đốc điều hành

income_data = np.concatenate([normal_income, outliers_error, outliers_legit])
df = pd.DataFrame({"Income": income_data})

print(f"Tổng số mẫu: {len(df)}")
print(f"Giá trị Min: {df['Income'].min():.2f} | Max: {df['Income'].max():.2f}")

# 2. Phương pháp 1: IQR Rule (Tukey)
Q1 = df["Income"].quantile(0.25)
Q3 = df["Income"].quantile(0.75)
IQR = Q3 - Q1
iqr_lower = Q1 - 1.5 * IQR
iqr_upper = Q3 + 1.5 * IQR
iqr_outliers = df[(df["Income"] < iqr_lower) | (df["Income"] > iqr_upper)]

# 3. Phương pháp 2: Z-Score truyền thống
mean = df["Income"].mean()
std = df["Income"].std()
z_scores = (df["Income"] - mean) / std
z_outliers = df[np.abs(z_scores) > 3]

# 4. Phương pháp 3: Modified Z-Score (MAD)
median = df["Income"].median()
mad = np.median(np.abs(df["Income"] - median))
modified_z_scores = 0.6745 * (df["Income"] - median) / mad
mod_z_outliers = df[np.abs(modified_z_scores) > 3.5]

print("\n=== KẾT QUẢ SO SÁNH PHÁT HIỆN NGOẠI LAI ===")
print(f"IQR Rule (Ngưỡng [{iqr_lower:.1f}, {iqr_upper:.1f}]): Tìm thấy {len(iqr_outliers)} ngoại lai")
print(f"Z-Score truyền thống (|Z| > 3): Tìm thấy {len(z_outliers)} ngoại lai (Bị Masking!)")
print(f"Modified Z-Score (|M| > 3.5): Tìm thấy {len(mod_z_outliers)} ngoại lai (Kháng Masking thành công)")

print("\nChi tiết các ngoại lai phát hiện bởi Modified Z-Score:")
print(mod_z_outliers)

# 5. Thực thi xử lý ngoại lai: Winsorization (Clipping)
# Ép toàn bộ giá trị về ngưỡng hợp lệ của IQR
df["Income_Cleaned_Clip"] = df["Income"].clip(lower=iqr_lower, upper=iqr_upper)

# Biến đổi Log Transformation (áp dụng trên phần giá trị dương)
df_positive = df[df["Income"] > 0].copy()
df_positive["Income_Log"] = np.log1p(df_positive["Income"])

print("\nThống kê mô tả sau khi Clipping:")
print(df["Income_Cleaned_Clip"].describe())

Tổng số mẫu: 104
Giá trị Min: -500.00 | Max: 99999.00

=== KẾT QUẢ SO SÁNH PHÁT HIỆN NGOẠI LAI ===
IQR Rule (Ngưỡng [815.1, 2160.9]): Tìm thấy 5 ngoại lai
Z-Score truyền thống (|Z| > 3): Tìm thấy 1 ngoại lai (Bị Masking!)
Modified Z-Score (|M| > 3.5): Tìm thấy 4 ngoại lai (Kháng Masking thành công)

Chi tiết các ngoại lai phát hiện bởi Modified Z-Score:
      Income
100   -500.0
101  99999.0
102   5500.0
103   6200.0

Thống kê mô tả sau khi Clipping:
count     104.000000
mean     1483.492206
std       296.389431
min       815.054984
25%      1319.728299
50%      1471.851239
75%      1656.177176
max      2160.850491
Name: Income_Cleaned_Clip, dtype: float64
